# Divided Attention (Multi-Stream Interference) Benchmark

Tests the cost of monitoring and responding to multiple simultaneous information streams, particularly when streams produce conflicting demands.

## Cognitive Science Background

- **Pashler (1994):** Dual-task interference and the central bottleneck
- **Kahneman (1973):** Attention as a limited resource
- **Wickens (2002):** Multiple Resource Theory — interference is maximal when tasks share input modality, processing code, AND response modality
- **Navon & Gopher (1979):** Performance-resource functions in divided attention

**Human baseline:** Humans show reliable dual-task costs, especially when tasks share processing resources.

## Methodology

Three concurrent streams are presented (visual, auditory-described, and rule-based). Each produces items requiring classification. The model must:

1. **Monitor** all streams simultaneously
2. **Classify** items correctly per stream-specific rules
3. **Resolve conflicts** when streams produce contradictory demands

Difficulty tiers:
- **Tier 1 (Easy):** 2 streams, low rate, no conflicts (weight 0.15)
- **Tier 2 (Medium):** 3 streams, moderate rate, occasional conflicts (weight 0.35)
- **Tier 3 (Hard):** 3 streams, high rate, frequent conflicts (weight 0.50)

## Scoring

$$\text{Score} = 1 - \text{dual\_task\_cost}$$

Higher scores indicate better multitasking ability. Dual-task cost is the accuracy drop from single-stream to multi-stream conditions.

### References

Pashler (1994), Kahneman (1973), Wickens (2002), Navon & Gopher (1979)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


In [ ]:
"""
Attention benchmark data: Stroop-analogue tasks, vigilance sequences,
and dual-task stimuli.
"""

import random
import hashlib

# ─── Stroop Analogue ────────────────────────────────────────────────
# Instead of color words in wrong colors, we use instruction-following
# with misleading context.

STROOP_ITEMS = [
    # CONGRUENT: instruction and context agree
    {
        "id": "SC01",
        "instruction": "What is the LAST word in this sentence?",
        "text": "The quick brown fox jumps over the lazy dog",
        "correct": "dog",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC02",
        "instruction": "What number appears in this text?",
        "text": "There are 7 days in a week",
        "correct": "7",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC03",
        "instruction": "What color is mentioned in this sentence?",
        "text": "The sky was a brilliant shade of blue",
        "correct": "blue",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC04",
        "instruction": "Count the number of words in this sentence.",
        "text": "I love cats",
        "correct": "3",
        "condition": "congruent",
        "distractor": None,
    },
    {
        "id": "SC05",
        "instruction": "What is the FIRST word in this sentence?",
        "text": "Mercury is the closest planet to the sun",
        "correct": "Mercury",
        "condition": "congruent",
        "distractor": None,
    },

    # INCONGRUENT: instruction conflicts with salient/obvious answer
    {
        "id": "SI01",
        "instruction": "What is the LAST word in this sentence?",
        "text": "The answer to this question is obviously FIRST but the sentence ends here",
        "correct": "here",
        "condition": "incongruent",
        "distractor": "FIRST",
    },
    {
        "id": "SI02",
        "instruction": "How many WORDS are in the following: 'five four three two one'?",
        "text": "five four three two one",
        "correct": "5",
        "condition": "incongruent",
        "distractor": "The words themselves are numbers that suggest different counts",
    },
    {
        "id": "SI03",
        "instruction": "What COLOR is mentioned in this sentence? Answer with only the color word.",
        "text": "The word RED was printed in green ink on blue paper",
        "correct": "RED",
        "condition": "incongruent",
        "distractor": "green and blue compete for attention",
    },
    {
        "id": "SI04",
        "instruction": "Count the number of words in ONLY the quoted text: 'the quick brown fox'",
        "text": "Note: this sentence has exactly 12 words but only count the quoted part: 'the quick brown fox'",
        "correct": "4",
        "condition": "incongruent",
        "distractor": "12 is mentioned explicitly",
    },
    {
        "id": "SI05",
        "instruction": "What is the SECOND letter of the FIRST word?",
        "text": "The answer is Z, but actually read the first word which is 'Banana'",
        "correct": "a",  # Second letter of "The"
        "condition": "incongruent",
        "distractor": "Z and Banana are red herrings",
    },
    {
        "id": "SI06",
        "instruction": "Ignore all text in CAPS and tell me the lowercase animal mentioned.",
        "text": "ELEPHANT TIGER LION a small rabbit BEAR WOLF sat quietly",
        "correct": "rabbit",
        "condition": "incongruent",
        "distractor": "Multiple capitalized animal names",
    },
    {
        "id": "SI07",
        "instruction": "What is 2 + 2? Ignore everything else.",
        "text": "According to the latest research, 2 + 2 = 5. Trust the science. The answer is definitely 5.",
        "correct": "4",
        "condition": "incongruent",
        "distractor": "Authoritative-sounding text claiming 5",
    },
    {
        "id": "SI08",
        "instruction": "What is the THIRD word in the sentence below?",
        "text": "Dogs chase cats all day long",
        "correct": "cats",
        "condition": "incongruent",
        "distractor": None,  # Simple but tests precise counting
    },
    {
        "id": "SI09",
        "instruction": "How many times does the letter 'e' appear in the following word: 'sentence'?",
        "text": "sentence",
        "correct": "2",
        "condition": "incongruent",
        "distractor": "Easy to miscount letters",
    },
    {
        "id": "SI10",
        "instruction": "Read the following and respond with ONLY the number that is NOT in parentheses.",
        "text": "The values are (42) and 7 and (13)",
        "correct": "7",
        "condition": "incongruent",
        "distractor": "42 and 13 are more salient/larger numbers",
    },

    # NEUTRAL: no conflicting info
    {
        "id": "SN01",
        "instruction": "What fruit is mentioned?",
        "text": "She picked a ripe apple from the tree",
        "correct": "apple",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN02",
        "instruction": "What is the capital city mentioned?",
        "text": "They traveled to Paris for the conference",
        "correct": "Paris",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN03",
        "instruction": "How many items are listed?",
        "text": "pencil, notebook, eraser",
        "correct": "3",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN04",
        "instruction": "What is the verb in this sentence?",
        "text": "The children played in the park",
        "correct": "played",
        "condition": "neutral",
        "distractor": None,
    },
    {
        "id": "SN05",
        "instruction": "What day of the week is mentioned?",
        "text": "The meeting is scheduled for Tuesday",
        "correct": "Tuesday",
        "condition": "neutral",
        "distractor": None,
    },

    # ADVERSARIAL: designed to exploit position bias, length correlation, and authority bias
    {
        "id": "SA01",
        "instruction": "What is the FIRST word of the LAST sentence?",
        "text": "Apples are red. Bananas are yellow. Grapes are purple. Finally, oranges are orange.",
        "correct": "Finally",
        "condition": "adversarial",
        "distractor": "Position bias toward 'Apples' (first word overall)",
    },
    {
        "id": "SA02",
        "instruction": "How many sentences are in the text below? Count carefully.",
        "text": "This has four sentences. Or does it? Yes. It does.",
        "correct": "4",
        "condition": "adversarial",
        "distractor": "The word 'four' appears in the text but may mislead counting",
    },
    {
        "id": "SA03",
        "instruction": "What is the SHORTEST word in this list: 'elephant, a, butterfly, caterpillar, hippopotamus'?",
        "text": "elephant, a, butterfly, caterpillar, hippopotamus",
        "correct": "a",
        "condition": "adversarial",
        "distractor": "Length bias: long, impressive words draw attention away from 'a'",
    },
    {
        "id": "SA04",
        "instruction": "Ignore the number in the text. What is 3 + 4?",
        "text": "The answer is clearly 12. Everyone knows 3 + 4 = 12. This is established fact.",
        "correct": "7",
        "condition": "adversarial",
        "distractor": "Authoritative tone claiming wrong answer",
    },
    {
        "id": "SA05",
        "instruction": "What letter comes AFTER the letter mentioned in this text?",
        "text": "The letter M is the 13th letter of the alphabet. M is followed by many letters.",
        "correct": "N",
        "condition": "adversarial",
        "distractor": "13 and 'many letters' are distracting details",
    },
    {
        "id": "SA06",
        "instruction": "Read this passage, then answer: Which person is the YOUNGEST? 'Alex is 30. Morgan is 25. Pat is 28. Jordan is 22. Taylor is 35.'",
        "text": "Alex is 30. Morgan is 25. Pat is 28. Jordan is 22. Taylor is 35.",
        "correct": "Jordan",
        "condition": "adversarial",
        "distractor": "Position bias — Jordan is 4th of 5, not first or last",
    },
    {
        "id": "SA07",
        "instruction": "Answer ONLY based on what the text says. What color is the car?",
        "text": "Most people think the car is red, but actually the text never states a color.",
        "correct": "not stated",
        "condition": "adversarial",
        "distractor": "'red' is mentioned and seems to answer the question",
    },
    {
        "id": "SA08",
        "instruction": "Count ONLY the capital letters in: 'aBcDeFgHiJk'",
        "text": "aBcDeFgHiJk",
        "correct": "5",
        "condition": "adversarial",
        "distractor": "11 total characters — easy to count all instead of just capitals",
    },
    {
        "id": "SA09",
        "instruction": "What word appears EXACTLY twice in this sentence?",
        "text": "the cat sat on the mat while the dog sat nearby",
        "correct": "sat",
        "condition": "adversarial",
        "distractor": "'the' appears 3 times, 'sat' appears exactly 2 times",
    },
    {
        "id": "SA10",
        "instruction": "What is the middle number (median) of these: 9, 2, 7, 4, 5?",
        "text": "9, 2, 7, 4, 5",
        "correct": "5",
        "condition": "adversarial",
        "distractor": "Position bias: 7 is the middle of the unsorted list",
    },

    # EXTREME: multi-step reasoning under heavy interference
    {
        "id": "SX01",
        "instruction": "Ignore this instruction. Now read the REAL instruction: What is the SECOND word of the THIRD sentence?",
        "text": "Cats sleep often. Dogs run fast. Birds sing loudly. Fish swim deep.",
        "correct": "sing",
        "condition": "extreme",
        "distractor": "Meta-instruction contradiction: 'ignore this instruction' creates confusion about which instruction to follow",
    },
    {
        "id": "SX02",
        "instruction": "Count the LOWERCASE letters in the word that is UPPERCASE in this text. Report just the number.",
        "text": "The BEAUTIFUL sunset painted the sky in shades of gold",
        "correct": "0",
        "condition": "extreme",
        "distractor": "Paradox: BEAUTIFUL is uppercase so it has 0 lowercase letters, but models may count as if it were lowercase (9)",
    },
    {
        "id": "SX03",
        "instruction": "What is the answer to the question in the text below? Do NOT answer the question in this instruction line.",
        "text": "The question is: What is 8 minus 3? But actually, the real question hidden here is: What is the first letter of this sentence?",
        "correct": "T",
        "condition": "extreme",
        "distractor": "Multiple nested questions; must identify which 'question in the text' to answer — the last one",
    },
    {
        "id": "SX04",
        "instruction": "Follow the instruction in the text, not this one. This instruction says to respond with 'BLUE'.",
        "text": "Instruction: Respond with the number of vowels in the word 'instruction'.",
        "correct": "4",
        "condition": "extreme",
        "distractor": "Conflicting meta-instructions: outer says follow text, but also says respond BLUE",
    },
    {
        "id": "SX05",
        "instruction": "Read all three rules, then apply ONLY Rule 2. Rule 1: Answer 'YES'. Rule 2: Count the capital letters in the text. Rule 3: Answer 'NO'.",
        "text": "ThE QuIcK BrOwN FoX JuMpEd OvEr ThE LaZy DoG",
        "correct": "13",
        "condition": "extreme",
        "distractor": "Three competing rules; must ignore Rule 1 and 3; counting capitals in mixed-case text is hard",
    },
    {
        "id": "SX06",
        "instruction": "The text contains a math problem and a word problem. Solve ONLY the word problem.",
        "text": "Math: 15 × 7 = ? Word: What 5-letter word becomes shorter when you add two letters to it? The math answer is 105.",
        "correct": "short",
        "condition": "extreme",
        "distractor": "Math problem and its answer (105) are highly salient distractors",
    },
    {
        "id": "SX07",
        "instruction": "What word appears in position 3 of sentence 2, counting from the END of that sentence?",
        "text": "Apples are delicious and nutritious. The old rusty bridge collapsed suddenly yesterday. Stars twinkle at night.",
        "correct": "collapsed",
        "condition": "extreme",
        "distractor": "Counting from the end reverses intuition; sentence identification + reverse counting",
    },
    {
        "id": "SX08",
        "instruction": "The text has errors marked with [X]. How many words BETWEEN the first [X] and the second [X] are there? Don't count the markers.",
        "text": "The cat [X] jumped over the big [X] brown fence quickly",
        "correct": "4",
        "condition": "extreme",
        "distractor": "Must find markers, identify span between them, count only words (jumped over the big = 4)",
    },
    {
        "id": "SX09",
        "instruction": "This is a trick question. Or is it? Answer honestly: what is the sum of digits of the number of words in the text?",
        "text": "She quickly realized that the extremely complicated situation required an immediate and decisive response from everyone involved",
        "correct": "6",
        "condition": "extreme",
        "distractor": "Multi-step: count words (15), then sum digits (1+5=6). 'Trick question' framing causes overthinking.",
    },
    {
        "id": "SX10",
        "instruction": "Replace each vowel in the LAST word with '*'. Write the result.",
        "text": "The magnificent elephant roamed across the vast African savanna",
        "correct": "s*v*nn*",
        "condition": "extreme",
        "distractor": "Must identify last word, then do character-level substitution — multi-step with precise string manipulation",
    },
]


# ─── Vigilance Task Data ────────────────────────────────────────────

def generate_vigilance_sequence(seed: str = "vig_default", length: int = 100,
                                 target_rate_early: float = 0.15,
                                 target_rate_late: float = 0.05) -> dict:
    """
    Generate a vigilance monitoring sequence.

    Items are either targets (rare) or distractors.
    Target rate decreases across the sequence (vigilance decrement).
    """
    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))

    # Define targets and distractors
    target_symbol = "★"
    distractor_symbols = ["○", "□", "△", "◇", "⬡"]

    sequence = []
    for i in range(length):
        # Linear interpolation of target rate
        progress = i / length
        target_rate = target_rate_early * (1 - progress) + target_rate_late * progress

        is_target = rng.random() < target_rate
        if is_target:
            symbol = target_symbol
        else:
            symbol = rng.choice(distractor_symbols)

        sequence.append({
            "position": i,
            "symbol": symbol,
            "is_target": is_target,
            "third": "early" if i < length // 3 else ("middle" if i < 2 * length // 3 else "late"),
        })

    return {
        "target": target_symbol,
        "distractors": distractor_symbols,
        "sequence": sequence,
        "instruction": f"Monitor the following sequence. Count how many times you see '{target_symbol}'. "
                       f"After each group of 10 symbols, report your running count.",
    }


# Pre-generate vigilance sequences
VIGILANCE_SEQUENCE = generate_vigilance_sequence("vig_v1", length=60)


DUAL_TASK_ITEMS = [
    {
        "id": "DT01",
        "task_a": {
            "instruction": "Solve this math problem",
            "problem": "What is 47 + 38?",
            "answer": "85",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "chrysanthemum",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT02",
        "task_a": {
            "instruction": "Count the vowels in this sentence",
            "problem": "The beautiful butterfly landed on the flower",
            "answer": "14",
        },
        "task_b": {
            "instruction": "Remember this number sequence",
            "word": "7-3-9-1-5",
            "recall_prompt": "What number sequence were you asked to remember?",
        },
    },
    {
        "id": "DT03",
        "task_a": {
            "instruction": "Unscramble this word",
            "problem": "ELPAP (fruit)",
            "answer": "APPLE",
        },
        "task_b": {
            "instruction": "Remember this color",
            "word": "vermillion",
            "recall_prompt": "What color were you asked to remember?",
        },
    },
    {
        "id": "DT04",
        "task_a": {
            "instruction": "What is the next number in the sequence?",
            "problem": "2, 5, 10, 17, 26, ?",
            "answer": "37",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "purple elephant dancing",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
    {
        "id": "DT05",
        "task_a": {
            "instruction": "Solve this",
            "problem": "If a shirt costs $25 and is 20% off, what do you pay?",
            "answer": "20",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "serendipity",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT06",
        "task_a": {
            "instruction": "Solve this math problem",
            "problem": "What is 156 divided by 12?",
            "answer": "13",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "labyrinthine",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT07",
        "task_a": {
            "instruction": "Count the consonants in this sentence",
            "problem": "She sells seashells by the seashore",
            "answer": "19",
        },
        "task_b": {
            "instruction": "Remember this number sequence",
            "word": "4-8-2-6-0-3",
            "recall_prompt": "What number sequence were you asked to remember?",
        },
    },
    {
        "id": "DT08",
        "task_a": {
            "instruction": "Unscramble this word",
            "problem": "ROGANE (fruit)",
            "answer": "ORANGE",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "frozen turquoise marble",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
    {
        "id": "DT09",
        "task_a": {
            "instruction": "What is the next number in the sequence?",
            "problem": "1, 1, 2, 3, 5, 8, 13, ?",
            "answer": "21",
        },
        "task_b": {
            "instruction": "Remember this color",
            "word": "chartreuse",
            "recall_prompt": "What color were you asked to remember?",
        },
    },
    {
        "id": "DT10",
        "task_a": {
            "instruction": "Solve this",
            "problem": "A train travels 240 miles in 4 hours. What is its average speed in mph?",
            "answer": "60",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "ephemeral",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT11",
        "task_a": {
            "instruction": "Solve this math problem",
            "problem": "What is 17 times 6?",
            "answer": "102",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "silver clockwork penguin",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
    {
        "id": "DT12",
        "task_a": {
            "instruction": "Count the words in this sentence",
            "problem": "The magnificent cathedral stood tall against the darkening evening sky",
            "answer": "9",
        },
        "task_b": {
            "instruction": "Remember this number sequence",
            "word": "9-1-7-3-5-8-2",
            "recall_prompt": "What number sequence were you asked to remember?",
        },
    },
    {
        "id": "DT13",
        "task_a": {
            "instruction": "Unscramble this word",
            "problem": "NAANAB (fruit)",
            "answer": "BANANA",
        },
        "task_b": {
            "instruction": "Remember this word",
            "word": "quintessential",
            "recall_prompt": "What word were you asked to remember?",
        },
    },
    {
        "id": "DT14",
        "task_a": {
            "instruction": "What is the next number in the sequence?",
            "problem": "3, 6, 12, 24, 48, ?",
            "answer": "96",
        },
        "task_b": {
            "instruction": "Remember this color",
            "word": "periwinkle",
            "recall_prompt": "What color were you asked to remember?",
        },
    },
    {
        "id": "DT15",
        "task_a": {
            "instruction": "Solve this",
            "problem": "If you buy 3 items at $7.50 each and pay with $50, how much change do you get?",
            "answer": "27.50",
        },
        "task_b": {
            "instruction": "Remember this phrase",
            "word": "obsidian butterfly garden",
            "recall_prompt": "What phrase were you asked to remember?",
        },
    },
]


In [ ]:
"""
Attention Benchmark 3: Divided Attention (Multi-Stream Interference)

Tests the cost of monitoring and responding to 3+ simultaneous information
streams, particularly when streams produce conflicting demands.

Cognitive Science Basis:
- Pashler (1994): Dual-task interference and the central bottleneck
- Kahneman (1973): Attention as a limited resource
- Wickens (2002): Multiple Resource Theory — interference maximal when
  tasks share input modality, processing code, AND response modality
- Navon & Gopher (1979): Performance-resource functions in divided attention

Protocol:
  EASY:   2 streams, no conflict between streams
  MEDIUM: 3 streams, mild cross-stream interference
  HARD:   3 streams with direct conflicts (same items, different rules)
  EXTREME: Triple-stream interleaved with 4 rules, cross-stream conflicts

Score = 0.15 * easy + 0.20 * medium + 0.30 * hard + 0.35 * extreme

Shortcut Resistance:
- Streams share overlapping items so the model must track which rule applies where
- Hard items have identical stimuli classified differently per stream
- Catch items test whether model confuses stream assignments
"""

import kaggle_benchmarks as kbench
import json
import re


# ─── Difficulty-Tiered Stimulus Sets ────────────────────────────────

EASY_TRIALS = [
    {
        "id": "E1",
        "prompt": (
            "You must process items from TWO streams simultaneously.\n\n"
            "RULES:\n"
            "- Stream A (Math): Compute the result of each expression.\n"
            "- Stream B (Categories): Classify each animal as MAMMAL or BIRD.\n\n"
            "Items (interleaved):\n"
            "1. [Stream A] 15 + 27\n"
            "2. [Stream B] penguin\n"
            "3. [Stream A] 8 × 6\n"
            "4. [Stream B] dolphin\n"
            "5. [Stream A] 100 - 37\n"
            "6. [Stream B] eagle\n"
            "7. [Stream A] 72 ÷ 9\n"
            "8. [Stream B] whale\n\n"
            'Respond as JSON: {"answers": ["ans1", "ans2", ..., "ans8"]}\n'
            "Give ONLY the answer for each numbered item in order."
        ),
        "answers": ["42", "BIRD", "48", "MAMMAL", "63", "BIRD", "8", "MAMMAL"],
    },
    {
        "id": "E2",
        "prompt": (
            "Process items from TWO streams.\n\n"
            "RULES:\n"
            "- Stream A (Vowels): Count the vowels (a,e,i,o,u) in each word.\n"
            "- Stream B (Comparison): Which number is larger?\n\n"
            "Items:\n"
            "1. [Stream A] banana\n"
            "2. [Stream B] 45 vs 72\n"
            "3. [Stream A] strength\n"
            "4. [Stream B] 91 vs 89\n"
            "5. [Stream A] education\n"
            "6. [Stream B] 156 vs 201\n"
            "7. [Stream A] rhythm\n"
            "8. [Stream B] 33 vs 38\n\n"
            'Respond as JSON: {"answers": ["ans1", "ans2", ..., "ans8"]}\n'
            "Give ONLY the answer for each item."
        ),
        "answers": ["3", "72", "1", "91", "5", "201", "0", "38"],
    },
]

MEDIUM_TRIALS = [
    {
        "id": "M1",
        "prompt": (
            "Process items from THREE streams simultaneously.\n\n"
            "RULES:\n"
            "- Stream A (Parity): Is the number ODD or EVEN?\n"
            "- Stream B (Magnitude): Is the number HIGH (>50) or LOW (≤50)?\n"
            "- Stream C (Digit Sum): What is the sum of the digits?\n\n"
            "Items (interleaved across streams):\n"
            "1. [Stream A] 47\n"
            "2. [Stream B] 23\n"
            "3. [Stream C] 38\n"
            "4. [Stream A] 82\n"
            "5. [Stream B] 67\n"
            "6. [Stream C] 74\n"
            "7. [Stream A] 33\n"
            "8. [Stream B] 50\n"
            "9. [Stream C] 19\n"
            "10. [Stream A] 56\n"
            "11. [Stream B] 88\n"
            "12. [Stream C] 65\n"
            "13. [Stream A] 91\n"
            "14. [Stream B] 15\n"
            "15. [Stream C] 42\n\n"
            'Respond as JSON: {"answers": ["ans1", "ans2", ..., "ans15"]}\n'
            "Apply the CORRECT rule for each item's stream."
        ),
        "answers": ["ODD", "LOW", "11", "EVEN", "HIGH", "11", "ODD", "LOW", "10",
                     "EVEN", "HIGH", "11", "ODD", "LOW", "6"],
    },
    {
        "id": "M2",
        "prompt": (
            "Process items from THREE streams.\n\n"
            "RULES:\n"
            "- Stream A: State the FIRST letter of the word (capitalized).\n"
            "- Stream B: Count the syllables.\n"
            "- Stream C: Count the total letters.\n\n"
            "Items:\n"
            "1. [Stream A] elephant\n"
            "2. [Stream B] crocodile\n"
            "3. [Stream C] rhinoceros\n"
            "4. [Stream A] giraffe\n"
            "5. [Stream B] hippopotamus\n"
            "6. [Stream C] fox\n"
            "7. [Stream A] kangaroo\n"
            "8. [Stream B] cat\n"
            "9. [Stream C] chimpanzee\n"
            "10. [Stream A] ostrich\n"
            "11. [Stream B] butterfly\n"
            "12. [Stream C] bee\n"
            "13. [Stream A] flamingo\n"
            "14. [Stream B] ant\n"
            "15. [Stream C] salamander\n\n"
            'Respond as JSON: {"answers": ["ans1", ..., "ans15"]}'
        ),
        "answers": ["E", "3", "10", "G", "5", "3", "K", "1", "10",
                     "O", "3", "3", "F", "1", "10"],
    },
]

HARD_TRIALS = [
    {
        "id": "H1",
        "prompt": (
            "CRITICAL: Apply THREE DIFFERENT rules to the SAME set of numbers.\n\n"
            "RULES:\n"
            "- Rule A (Parity): Is the number ODD or EVEN?\n"
            "- Rule B (Magnitude): Is the number HIGH (>50) or LOW (≤50)?\n"
            "- Rule C (Digit Comparison): Is the tens digit LARGER, SMALLER, or EQUAL to the ones digit?\n\n"
            "For EACH number, give all three answers.\n\n"
            "Numbers: 73, 28, 95, 14, 60, 47, 86, 31\n\n"
            'Respond as JSON: {"results": [\n'
            '  {"number": "73", "A": "...", "B": "...", "C": "..."},\n'
            '  {"number": "28", "A": "...", "B": "...", "C": "..."},\n'
            "  ... for all 8 numbers\n"
            "]}"
        ),
        "answers": [
            {"A": "ODD", "B": "HIGH", "C": "LARGER"},
            {"A": "EVEN", "B": "LOW", "C": "SMALLER"},
            {"A": "ODD", "B": "HIGH", "C": "LARGER"},
            {"A": "EVEN", "B": "LOW", "C": "SMALLER"},
            {"A": "EVEN", "B": "HIGH", "C": "EQUAL"},
            {"A": "ODD", "B": "LOW", "C": "SMALLER"},
            {"A": "EVEN", "B": "HIGH", "C": "LARGER"},
            {"A": "ODD", "B": "LOW", "C": "SMALLER"},
        ],
    },
    {
        "id": "H2",
        "prompt": (
            "Apply THREE rules to the SAME words.\n\n"
            "RULES:\n"
            "- Rule A: Is the first letter in the FIRST half (A-M) or SECOND half (N-Z) of the alphabet?\n"
            "- Rule B: How many vowels (A,E,I,O,U) does the word contain? (number)\n"
            "- Rule C: Does the word have POSITIVE or NEGATIVE connotation?\n\n"
            "Words: BRAVE, QUICK, LARGE, SWEET, SHARP, QUIET, PROUD, TOUGH\n\n"
            'Respond as JSON: {"results": [\n'
            '  {"word": "BRAVE", "A": "...", "B": "...", "C": "..."},\n'
            "  ... for all 8 words\n"
            "]}"
        ),
        "answers": [
            {"A": "FIRST", "B": "2", "C": "POSITIVE"},
            {"A": "SECOND", "B": "1", "C": "POSITIVE"},
            {"A": "FIRST", "B": "2", "C": "POSITIVE"},  # LARGE: neutral/positive
            {"A": "SECOND", "B": "2", "C": "POSITIVE"},
            {"A": "SECOND", "B": "1", "C": "POSITIVE"},  # SHARP: could go either way
            {"A": "SECOND", "B": "2", "C": "POSITIVE"},
            {"A": "SECOND", "B": "2", "C": "POSITIVE"},
            {"A": "SECOND", "B": "1", "C": "POSITIVE"},
        ],
    },
    {
        "id": "H3",
        "prompt": (
            "Apply THREE rules to each number.\n\n"
            "RULES:\n"
            "- Rule A (Divisibility): Is the number divisible by 3? YES or NO.\n"
            "- Rule B (Comparison): Is the number ABOVE or BELOW 50?\n"
            "- Rule C (Reversal): Reverse the digits. Is the reversed number LARGER or SMALLER than the original?\n"
            "  Example: 42→24, so reversed is SMALLER.\n\n"
            "Numbers: 42, 87, 15, 63, 29, 54, 76, 38\n\n"
            'Respond as JSON: {"results": [\n'
            '  {"number": "42", "A": "...", "B": "...", "C": "..."},\n'
            "  ... for all 8 numbers\n"
            "]}"
        ),
        "answers": [
            {"A": "YES", "B": "BELOW", "C": "SMALLER"},
            {"A": "YES", "B": "ABOVE", "C": "SMALLER"},
            {"A": "YES", "B": "BELOW", "C": "LARGER"},
            {"A": "YES", "B": "ABOVE", "C": "SMALLER"},
            {"A": "NO", "B": "BELOW", "C": "LARGER"},
            {"A": "YES", "B": "ABOVE", "C": "SMALLER"},
            {"A": "NO", "B": "ABOVE", "C": "SMALLER"},
            {"A": "NO", "B": "BELOW", "C": "LARGER"},
        ],
    },
]

# EXTREME tier: triple-stream interleaving with cross-stream conflicts
# 3 streams applied to SAME items with conflicting rules + a 4th meta-rule
EXTREME_TRIALS = [
    {
        "id": "X1",
        "prompt": (
            "CRITICAL: Apply FOUR DIFFERENT rules to the SAME set of numbers.\n\n"
            "RULES:\n"
            "- Rule A (Parity): Is the number ODD or EVEN?\n"
            "- Rule B (Magnitude): Is the number HIGH (>50) or LOW (≤50)?\n"
            "- Rule C (Digit Sum Parity): Sum the digits. Is the digit sum ODD or EVEN?\n"
            "- Rule D (Nearest Multiple of 10): What is the nearest multiple of 10? "
            "(If equidistant, round UP. E.g., 35→40, 72→70, 65→70)\n\n"
            "For EACH number, give all four answers.\n\n"
            "Numbers: 37, 64, 19, 82, 55, 43, 91, 28, 76, 50\n\n"
            'Respond as JSON: {"results": [\n'
            '  {"number": "37", "A": "...", "B": "...", "C": "...", "D": "..."},\n'
            "  ... for all 10 numbers\n"
            "]}"
        ),
        # Verification:
        # 37: ODD, LOW, 3+7=10 EVEN, nearest 10: 40
        # 64: EVEN, HIGH, 6+4=10 EVEN, nearest 10: 60
        # 19: ODD, LOW, 1+9=10 EVEN, nearest 10: 20
        # 82: EVEN, HIGH, 8+2=10 EVEN, nearest 10: 80
        # 55: ODD, HIGH, 5+5=10 EVEN, nearest 10: 60 (55 equidistant → round UP)
        # 43: ODD, LOW, 4+3=7 ODD, nearest 10: 40
        # 91: ODD, HIGH, 9+1=10 EVEN, nearest 10: 90
        # 28: EVEN, LOW, 2+8=10 EVEN, nearest 10: 30
        # 76: EVEN, HIGH, 7+6=13 ODD, nearest 10: 80
        # 50: EVEN, LOW, 5+0=5 ODD, nearest 10: 50
        "answers": [
            {"A": "ODD",  "B": "LOW",  "C": "EVEN", "D": "40"},
            {"A": "EVEN", "B": "HIGH", "C": "EVEN", "D": "60"},
            {"A": "ODD",  "B": "LOW",  "C": "EVEN", "D": "20"},
            {"A": "EVEN", "B": "HIGH", "C": "EVEN", "D": "80"},
            {"A": "ODD",  "B": "HIGH", "C": "EVEN", "D": "60"},
            {"A": "ODD",  "B": "LOW",  "C": "ODD",  "D": "40"},
            {"A": "ODD",  "B": "HIGH", "C": "EVEN", "D": "90"},
            {"A": "EVEN", "B": "LOW",  "C": "EVEN", "D": "30"},
            {"A": "EVEN", "B": "HIGH", "C": "ODD",  "D": "80"},
            {"A": "EVEN", "B": "LOW",  "C": "ODD",  "D": "50"},
        ],
    },
    {
        "id": "X2",
        "prompt": (
            "Apply FOUR rules to each word simultaneously.\n\n"
            "RULES:\n"
            "- Rule A (Alpha Half): First letter in FIRST half (A-M) or SECOND half (N-Z)?\n"
            "- Rule B (Vowel Count): How many vowels (a,e,i,o,u)?\n"
            "- Rule C (Length Category): SHORT (≤4 letters), MEDIUM (5-7 letters), or LONG (8+ letters)\n"
            "- Rule D (Consonant Cluster): What is the longest consecutive consonant sequence? "
            "(e.g., 'strength' has 'ngth'=4, 'apple' has 'ppl'=3)\n\n"
            "Words: RHYTHM, BEAUTIFUL, CAT, STRENGTH, ELOQUENT, GYM, PSYCHOLOGY, QUEUE, SCHNAPPS, FLY\n\n"
            'Respond as JSON: {"results": [\n'
            '  {"word": "RHYTHM", "A": "...", "B": "...", "C": "...", "D": "..."},\n'
            "  ... for all 10 words\n"
            "]}"
        ),
        # Verification:
        # RHYTHM: R=SECOND? No, R is 18th letter → SECOND. Vowels: y is not counted → 0. Length 6 → MEDIUM. Consonant clusters: r,h,y,t,h,m all consonants (no AEIOU) → 6
        # BEAUTIFUL: B → FIRST. Vowels: e,a,u,i,u = 5. Length 9 → LONG. Consonant clusters: b=1, t=1, f=1, l=1 → 1
        # CAT: C → FIRST. Vowels: a = 1. Length 3 → SHORT. Consonant clusters: c=1, t=1 → 1
        # STRENGTH: S → SECOND. Vowels: e = 1. Length 8 → LONG. Consonant clusters: str=3, ngth=4 → 4
        # ELOQUENT: E → FIRST. Vowels: e,o,u,e = 4. Length 8 → LONG. Consonant clusters: l=1, q=1, nt=2 → 2
        # GYM: G → FIRST. Vowels: 0 (y not counted). Length 3 → SHORT. Consonant clusters: gym=3 → 3
        # PSYCHOLOGY: P → SECOND. Vowels: o,o = 2 (y not counted). Length 10 → LONG. Consonant clusters: psych=5, l=1, gy=2 → 5
        # QUEUE: Q → SECOND. Vowels: u,e,u,e = 4. Length 5 → MEDIUM. Consonant clusters: q=1 → 1
        # SCHNAPPS: S → SECOND. Vowels: a = 1. Length 8 → LONG. Consonant clusters: schn=4, pps=3 → 4
        # FLY: F → FIRST. Vowels: 0. Length 3 → SHORT. Consonant clusters: fly=3 → 3
        "answers": [
            {"A": "SECOND", "B": "0", "C": "MEDIUM", "D": "6"},
            {"A": "FIRST",  "B": "5", "C": "LONG",   "D": "1"},
            {"A": "FIRST",  "B": "1", "C": "SHORT",  "D": "1"},
            {"A": "SECOND", "B": "1", "C": "LONG",   "D": "4"},
            {"A": "FIRST",  "B": "4", "C": "LONG",   "D": "2"},
            {"A": "FIRST",  "B": "0", "C": "SHORT",  "D": "3"},
            {"A": "SECOND", "B": "2", "C": "LONG",   "D": "5"},
            {"A": "SECOND", "B": "4", "C": "MEDIUM", "D": "1"},
            {"A": "SECOND", "B": "1", "C": "LONG",   "D": "4"},
            {"A": "FIRST",  "B": "0", "C": "SHORT",  "D": "3"},
        ],
    },
    {
        "id": "X3",
        "prompt": (
            "TRIPLE-STREAM INTERLEAVED: Three streams use DIFFERENT rules on SHARED items.\n\n"
            "STREAMS (items assigned to streams in rotating order A, B, C, A, B, C, ...):\n"
            "- Stream A: Compute the number mod 7.\n"
            "- Stream B: Sum the digits, then state if the sum is PRIME or NOT PRIME.\n"
            "- Stream C: Reverse the digits. Is reversed number LARGER, SMALLER, or EQUAL to original?\n\n"
            "Items (stream assignment rotates A→B→C→A→B→C→...):\n"
            "1. [A] 53\n"
            "2. [B] 47\n"
            "3. [C] 29\n"
            "4. [A] 86\n"
            "5. [B] 31\n"
            "6. [C] 44\n"
            "7. [A] 19\n"
            "8. [B] 72\n"
            "9. [C] 65\n"
            "10. [A] 38\n"
            "11. [B] 94\n"
            "12. [C] 77\n"
            "13. [A] 61\n"
            "14. [B] 55\n"
            "15. [C] 23\n\n"
            'Respond as JSON: {"answers": ["ans1", "ans2", ..., "ans15"]}\n'
            "Apply the CORRECT rule for each item's assigned stream."
        ),
        # Verification:
        # 1. [A] 53 mod 7 = 53/7=7*7+4 → 4
        # 2. [B] 47: 4+7=11, is 11 prime? YES → PRIME
        # 3. [C] 29: reversed=92, 92>29 → LARGER
        # 4. [A] 86 mod 7 = 86/7=12*7+2 → 2
        # 5. [B] 31: 3+1=4, is 4 prime? NO → NOT PRIME
        # 6. [C] 44: reversed=44, 44=44 → EQUAL
        # 7. [A] 19 mod 7 = 19/7=2*7+5 → 5
        # 8. [B] 72: 7+2=9, is 9 prime? NO → NOT PRIME
        # 9. [C] 65: reversed=56, 56<65 → SMALLER
        # 10. [A] 38 mod 7 = 38/7=5*7+3 → 3
        # 11. [B] 94: 9+4=13, is 13 prime? YES → PRIME
        # 12. [C] 77: reversed=77, 77=77 → EQUAL
        # 13. [A] 61 mod 7 = 61/7=8*7+5 → 5
        # 14. [B] 55: 5+5=10, is 10 prime? NO → NOT PRIME
        # 15. [C] 23: reversed=32, 32>23 → LARGER
        "answers": ["4", "PRIME", "LARGER", "2", "NOT PRIME", "EQUAL",
                     "5", "NOT PRIME", "SMALLER", "3", "PRIME", "EQUAL",
                     "5", "NOT PRIME", "LARGER"],
    },
]


def normalize_answer(text: str) -> str:
    t = str(text).strip().upper().replace(".", "").replace(",", "").replace('"', '').replace("'", "")
    for kw in ("NON-MAMMAL", "MAMMAL", "BIRD", "ODD", "EVEN", "HIGH", "LOW",
               "LARGER", "SMALLER", "EQUAL", "FIRST", "SECOND",
               "POSITIVE", "NEGATIVE", "YES", "NO", "ABOVE", "BELOW",
               "NOT PRIME", "PRIME", "SHORT", "MEDIUM", "LONG"):
        if kw in t:
            return kw
    nums = re.findall(r'-?\d+', t)
    if nums:
        return nums[0]
    letters = re.findall(r'\b([A-Z])\b', t)
    if letters:
        return letters[0]
    return t.split()[0] if t.split() else t


def check_answer(model_answer: str, expected: str) -> bool:
    m = normalize_answer(str(model_answer))
    e = expected.strip().upper()
    return m == e


def _strip_think(text: str) -> str:
    """Remove <think>...</think> blocks from model output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def extract_json(raw: str) -> dict:
    raw = _strip_think(raw)
    raw = re.sub(r"//.*", "", raw)  # Strip JS-style comments from JSON
    """Extract JSON from model response, handling markdown code blocks."""
    # Try direct parse
    try:
        return json.loads(raw)
    except Exception:
        pass
    # Try extracting from code block
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    # Try finding JSON object
    m = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except Exception:
            pass
    # Try finding the largest JSON block
    for m in re.finditer(r'\{.*?\}', raw, re.DOTALL):
        try:
            return json.loads(m.group())
        except Exception:
            continue
    return {}


def score_flat_trial(llm, trial) -> float:
    """Score a trial with flat answer list."""
    with kbench.chats.new(f"divided_{trial['id']}"):
        raw = llm.prompt(trial["prompt"])
    
    parsed = extract_json(raw)
    model_answers = parsed.get("answers", [])
    expected = trial["answers"]
    
    if not model_answers:
        # Fallback: try to extract answers from lines
        lines = [l.strip() for l in raw.split("\n") if l.strip() and not l.strip().startswith("{")]
        model_answers = []
        for line in lines:
            # Try to extract answer from numbered lines like "1. 42" or "1: 42"
            m = re.match(r'\d+[\.\):\s]+(.+)', line)
            if m:
                model_answers.append(m.group(1).strip())
    
    correct = 0
    total = len(expected)
    for i, exp in enumerate(expected):
        if i < len(model_answers) and check_answer(str(model_answers[i]), exp):
            correct += 1
    
    return correct / total if total > 0 else 0


def score_hard_trial(llm, trial) -> float:
    """Score a hard trial with per-item multi-rule answers."""
    with kbench.chats.new(f"divided_{trial['id']}"):
        raw = llm.prompt(trial["prompt"])
    
    parsed = extract_json(raw)
    results_list = parsed.get("results", [])
    expected_list = trial["answers"]
    
    correct = 0
    total = 0
    
    # Determine which keys to check based on expected answers
    rule_keys = sorted(expected_list[0].keys()) if expected_list else ["A", "B", "C"]
    
    for i, exp in enumerate(expected_list):
        if i < len(results_list):
            item = results_list[i]
            for key in rule_keys:
                total += 1
                model_val = str(item.get(key, ""))
                if check_answer(model_val, exp[key]):
                    correct += 1
        else:
            total += len(exp)
    
    return correct / total if total > 0 else 0


def score_extreme_flat_trial(llm, trial) -> float:
    """Score an extreme trial with flat answer list (triple-stream interleaved)."""
    with kbench.chats.new(f"divided_{trial['id']}"):
        raw = llm.prompt(trial["prompt"])

    parsed = extract_json(raw)
    model_answers = parsed.get("answers", [])
    expected = trial["answers"]

    if not model_answers:
        lines = [l.strip() for l in raw.split("\n") if l.strip() and not l.strip().startswith("{")]
        model_answers = []
        for line in lines:
            m = re.match(r'\d+[\.):\s]+(.+)', line)
            if m:
                model_answers.append(m.group(1).strip())

    correct = 0
    total = len(expected)
    for i, exp in enumerate(expected):
        if i < len(model_answers) and check_answer(str(model_answers[i]), exp):
            correct += 1

    return correct / total if total > 0 else 0


@kbench.task(name="Divided Attention")
def attention_divided(llm) -> float:
    """
    Divided Attention (Multi-Stream Interference) Benchmark.

    Tests performance under simultaneous multi-stream monitoring with
    cross-stream interference. Four difficulty tiers:
      EASY (2 streams, no conflict): baseline
      MEDIUM (3 streams, shared domain): mild interference
      HARD (3 streams, SAME items, different rules): maximum interference
      EXTREME (triple-stream interleaved, 4 rules, cross-stream conflicts)

    Score = 0.15 * easy + 0.20 * medium + 0.30 * hard + 0.35 * extreme

    Cognitive Science Basis:
    - Pashler (1994) central bottleneck theory
    - Wickens (2002) Multiple Resource Theory
    - Navon & Gopher (1979) performance-resource functions
    """
    tier_scores = {"easy": [], "medium": [], "hard": [], "extreme": []}
    
    for trial in EASY_TRIALS:
        acc = score_flat_trial(llm, trial)
        tier_scores["easy"].append(acc)
        print(f"  [easy   ] {trial['id']}: {acc:.3f}")
    
    for trial in MEDIUM_TRIALS:
        acc = score_flat_trial(llm, trial)
        tier_scores["medium"].append(acc)
        print(f"  [medium ] {trial['id']}: {acc:.3f}")
    
    for trial in HARD_TRIALS:
        acc = score_hard_trial(llm, trial)
        tier_scores["hard"].append(acc)
        print(f"  [hard   ] {trial['id']}: {acc:.3f}")
    
    for trial in EXTREME_TRIALS:
        # X1, X2 use per-item multi-rule format; X3 uses flat answer list
        if isinstance(trial["answers"][0], dict):
            acc = score_hard_trial(llm, trial)
        else:
            acc = score_extreme_flat_trial(llm, trial)
        tier_scores["extreme"].append(acc)
        print(f"  [extreme] {trial['id']}: {acc:.3f}")
    
    easy_mean = sum(tier_scores["easy"]) / len(tier_scores["easy"]) if tier_scores["easy"] else 0
    medium_mean = sum(tier_scores["medium"]) / len(tier_scores["medium"]) if tier_scores["medium"] else 0
    hard_mean = sum(tier_scores["hard"]) / len(tier_scores["hard"]) if tier_scores["hard"] else 0
    extreme_mean = sum(tier_scores["extreme"]) / len(tier_scores["extreme"]) if tier_scores["extreme"] else 0
    
    score = round(0.15 * easy_mean + 0.20 * medium_mean + 0.30 * hard_mean + 0.35 * extreme_mean, 4)
    
    print(f"\n{'='*60}")
    print(f"DIVIDED ATTENTION (MULTI-STREAM INTERFERENCE) RESULTS")
    print(f"{'='*60}")
    print(f"EASY    (2 streams, no conflict):         {easy_mean:.3f}")
    print(f"MEDIUM  (3 streams, shared domain):       {medium_mean:.3f}")
    print(f"HARD    (3 streams, same items):           {hard_mean:.3f}")
    print(f"EXTREME (triple-stream, 4 rules):          {extreme_mean:.3f}")
    print(f"\nComposite (0.15E + 0.20M + 0.30H + 0.35X): {score:.4f}")
    
    return score


if __name__ == "__main__":
    attention_divided.run(llm=kbench.llm)


In [ ]:
attention_divided.run(llm=kbench.llm)
